In [ ]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns

os.makedirs("figures", exist_ok=True)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


In [ ]:
adata = sc.read_h5ad('adata_anno_cell_subtype_re.h5ad')

In [ ]:
adata_normal = adata[adata.obs['status']=='normal-like']
adata_tumor = adata[adata.obs['status']=='tumor']

In [ ]:


cell_counts = (
    adata_normal.obs
    .groupby(['sample', 'cell_type'])
    .size()
    .reset_index(name='count')
)


total_counts = (
    adata_normal.obs.groupby('sample').size().reset_index(name='total_count')
)


cell_counts = cell_counts.merge(total_counts, on='sample')
cell_counts['proportion'] = cell_counts['count'] / cell_counts['total_count']

cell_type_order = list(adata_normal.uns['cell_type_colors'].keys()) \
    if isinstance(adata_normal.uns['cell_type_colors'], dict) \
    else adata_normal.obs['cell_type'].cat.categories.tolist()
palette = adata_normal.uns['cell_type_colors']


plt.figure(figsize=(5, 5))

sns.boxplot(
    data=cell_counts,
    x='cell_type',
    y='proportion',
    palette=palette,
    order=cell_type_order,
    showfliers=False,
    width=0.6
)



plt.ylabel('Proportion in each sample')
plt.xlabel('Cell type')
plt.title('normal sample')
plt.xticks(rotation=90)
plt.ylim(-0.05, 1.05)
plt.yticks([0, 0.25, 0.5, 0.75, 1.0])
plt.tight_layout()
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.savefig('figures/boxplot_normal.pdf',bbox_inches='tight')
plt.show()


In [ ]:


cell_counts = (
    adata_tumor.obs
    .groupby(['sample', 'cell_type'])
    .size()
    .reset_index(name='count')
)


total_counts = (
    adata_tumor.obs.groupby('sample').size().reset_index(name='total_count')
)


cell_counts = cell_counts.merge(total_counts, on='sample')
cell_counts['proportion'] = cell_counts['count'] / cell_counts['total_count']


cell_type_order = list(adata_tumor.uns['cell_type_colors'].keys()) \
    if isinstance(adata_tumor.uns['cell_type_colors'], dict) \
    else adata_tumor.obs['cell_type'].cat.categories.tolist()
palette = adata_tumor.uns['cell_type_colors']


plt.figure(figsize=(5, 5))

sns.boxplot(
    data=cell_counts,
    x='cell_type',
    y='proportion',
    palette=palette,
    order=cell_type_order,
    showfliers=False,
    width=0.6
)

plt.ylabel('Proportion in each sample')
plt.xlabel('Cell type')
plt.title('tumor sample')
plt.xticks(rotation=90)
plt.ylim(-0.05, 1.05)
plt.yticks([0, 0.25, 0.5, 0.75, 1.0])
plt.tight_layout()
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.savefig('figures/boxplot_tumor.pdf',bbox_inches='tight')
plt.show()


In [ ]:
import warnings
import rapids_singlecell as rsc
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

warnings.filterwarnings('ignore')
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

sc.set_figure_params(figsize=(3.5, 3.5), dpi=300)
os.makedirs("figures/8_unintegrated_series_umap", exist_ok=True)
sc.settings.figdir = "figures/8_unintegrated_series_umap"


In [ ]:
adata = sc.read_h5ad("adata_qc.h5ad")


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata, n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata, max_value=10)
rsc.tl.pca(adata, n_comps=50)
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=15)
rsc.tl.umap(adata, min_dist=0.5, spread=1.0, n_components=2)
rsc.get.anndata_to_CPU(adata)
adata.write_h5ad("adata_qc_unintegrated_before_harmony_umap.h5ad")


In [ ]:
adata_unintegrated = sc.read_h5ad("adata_qc_unintegrated_before_harmony_umap.h5ad")


In [ ]:
sc.pl.umap(
    adata_unintegrated,
    color="series",
    frameon=True,
    size=0.2,
    show=False,
    save="_adata_qc_unintegrated_saved_series_size0p2.pdf",
)


In [ ]:
adata_anno = sc.read_h5ad("adata_anno_cell_subtype_re.h5ad")


In [ ]:
sc.pl.umap(
    adata_anno,
    color="series",
    frameon=True,
    size=0.2,
    show=False,
    save="_adata_anno_integrated_series_size0p2.pdf",
)
